In [4]:
import pandas as pd

sales = pd.read_csv('Megastore Dataset.csv')
tx = (sales.groupby('OrderID')['ProductName']
      .apply(lambda products: sorted(set(products.dropna())))
      .reset_index(name='Transaction'))

print(f'{len(tx)} transactions loaded')

441 transactions loaded


In [8]:
from itertools import combinations

transactions = tx['Transaction'].tolist()          # 441 baskets
n_tx = len(transactions)
tx_sets = [set(t) for t in transactions]

def support_count(itemset):                        # baskets containing itemset
    return sum(1 for t in tx_sets if itemset.issubset(t))

def apriori(transactions, min_support, max_k=4):   # (Agrawal & Srikant, 1994)
    items = sorted(set(i for t in transactions for i in t))
    L = {1: [c for c in [(i,) for i in items]
             if support_count(set(c)) / n_tx >= min_support]}
    k = 2
    while L.get(k - 1) and k <= max_k:
        prev = L[k - 1]; cand = set()
        for i in range(len(prev)):                 # JOIN step
            for j in range(i + 1, len(prev)):
                if prev[i][:k-2] == prev[j][:k-2]:
                    cand.add(tuple(sorted(set(prev[i]) | set(prev[j]))))
        freq = []
        for c in cand:                             # PRUNE step (Apriori property)
            if all(tuple(sorted(s)) in set(prev) for s in combinations(c, k-1)):
                if support_count(set(c)) / n_tx >= min_support:
                    freq.append(c)
        L[k] = sorted(freq); k += 1
    return L

freq_itemsets = apriori(transactions, min_support=0.03, max_k=3)
sup_map = {frozenset(its): support_count(set(its)) / n_tx
           for k, v in freq_itemsets.items() for its in v}

rules = []
for k, v in freq_itemsets.items():                 # RULE generation (Han et al., 2011)
    if k < 2: continue
    for its in v:
        ab = frozenset(its); s_ab = sup_map[ab]
        for r in range(1, k):
            for a in combinations(its, r):
                A, B = frozenset(a), ab - frozenset(a)
                conf = s_ab / sup_map[A]
                if conf >= 0.50:
                    rules.append({'antecedent': ', '.join(sorted(A)),
                                  'consequent': ', '.join(sorted(B)),
                                  'support': round(s_ab, 4),
                                  'confidence': round(conf, 4),
                                  'lift': round(conf / sup_map[B], 4)})
rules = pd.DataFrame(rules).drop_duplicates(
    subset=['antecedent', 'consequent']).sort_values(
    ['lift', 'confidence'], ascending=False)       # sorted by chosen metric: LIFT

# Keep one best directed rule for each underlying item association.
rules['itemset_key'] = rules.apply(
    lambda row: tuple(sorted(row['antecedent'].split(', ') +
                             row['consequent'].split(', '))), axis=1)
top3_rules = (rules.drop_duplicates('itemset_key')
                   .drop(columns='itemset_key')
                   .head(3))
top3_rules.to_csv('Top3_Association_Rules.csv', index=False)
print('Exported:', 'Top3_Association_Rules.csv')

Exported: Top3_Association_Rules.csv


In [12]:
# ---------- ORDINAL ENCODING ----------
df = sales.copy()

# OrderPriority: Medium=1, High=2  (preserves urgency order)
priority_map = {'Medium': 1, 'High': 2}
df['OrderPriority_Encoded'] = df['OrderPriority'].map(priority_map)

# CustomerOrderSatisfaction: Likert 1-4; 'Prefer not to answer' = 0
# (no valid rank -> neutral/missing indicator, not fabricated rank)
sat_map = {'Very Dissatisfied': 1, 'Dissatisfied': 2, 'Satisfied': 3,
           'Very Satisfied': 4, 'Prefer not to answer': 0}
df['CustomerSatisfaction_Encoded'] = df['CustomerOrderSatisfaction'].map(sat_map)

# ---------- ONE-HOT ENCODING (nominal) ----------
df = pd.get_dummies(df, columns=['Segment', 'Region'], dtype=int)
# Produces: Segment_Consumer, Segment_Corporate, Region_Northeast, Region_Southeast

df.to_csv('Megastore_Encoded.csv', index=False)
print('Exported:', 'Megastore_Encoded.csv')

Exported: Megastore_Encoded.csv


In [15]:
tx_products = (sales.groupby('OrderID')
               .agg(ProductName=('ProductName', lambda x: sorted(set(x.dropna()))),
                    Segment=('Segment', 'first'),
                    Region=('Region', 'first'))
               .reset_index())

tx_products['Transaction'] = tx_products.apply(
    lambda row: row['ProductName'] +
                [f"Segment_{row['Segment']}", f"Region_{row['Region']}"],
    axis=1)
tx = tx_products[['OrderID', 'Transaction']]

tx.to_csv('Megastore_Transactions.csv', index=False)
print('Exported:', 'Megastore_Transactions.csv')

Exported: Megastore_Transactions.csv


# Analysis Explanation and Software Version

## Purpose
This notebook analyzes the megastore sales data in `Megastore Dataset.csv`. It prepares transaction-level data, finds frequent product combinations, generates association rules, encodes selected categorical variables, and exports the results as CSV files.

## 1. Data loading and transaction preparation
The dataset is loaded into the `sales` DataFrame with pandas. Rows are grouped by `OrderID`, so each order becomes one transaction or basket. The distinct non-missing `ProductName` values in each order are stored in the `Transaction` column. The transaction table is exported as `Megastore_Transactions.csv`.

The transaction-building cell also appends the order's segment and region as item labels such as `Segment_Corporate` and `Region_Northeast`. This allows these attributes to be analyzed alongside products in the association-rule analysis.

## 2. Apriori frequent-itemset analysis
The Apriori algorithm identifies item combinations that occur frequently in the transactions.

- `support_count()` counts how many baskets contain an itemset.
- Candidate itemsets are generated by joining smaller frequent itemsets.
- The Apriori pruning step removes candidates whose smaller subsets are not frequent.
- The minimum support is `0.03`, and combinations up to three items are examined.

For each frequent itemset, association rules are generated. Rules are retained when confidence is at least `0.50`. The reported metrics are:

- **Support:** proportion of baskets containing both the antecedent and consequent.
- **Confidence:** probability of seeing the consequent when the antecedent occurs.
- **Lift:** how much more often the consequent occurs with the antecedent than would be expected independently.

Rules are sorted by lift and confidence. Reverse directions of the same underlying association are collapsed so the exported file contains the top three distinct associations in `Top3_Association_Rules.csv`.

## 3. Feature encoding
A copy of the sales data is stored in `df`. The ordinal variables are mapped to numeric values:

- `OrderPriority`: Medium = 1, High = 2
- `CustomerOrderSatisfaction`: Very Dissatisfied = 1 through Very Satisfied = 4; Prefer not to answer = 0

`Segment` and `Region` are nominal variables, so they are converted into one-hot encoded columns with `pd.get_dummies()`. The encoded dataset is exported as `Megastore_Encoded.csv`.

## Software and programming language
The analysis is written in **Python 3** and uses **pandas** for data manipulation. The exact interpreter and pandas versions are reported by the next code cell.

In [16]:
import platform
import pandas as pd

print(f'Python version: {platform.python_version()}')
print(f'pandas version: {pd.__version__}')

Python version: 3.13.14
pandas version: 3.0.5
